# LangGraph 001 — Getting Ready

An orientation lesson, so this notebook checks that you are ready: your
packages, your first graph, and the three pieces of Python the track leans on.

| Part | What we check |
|---|---|
| A | package versions, and a one-node graph that runs |
| B | `TypedDict` — how LangGraph describes state |
| C | Pydantic — state that checks its own types |
| D | `asyncio` — running two slow calls at once |

Needs `pip install langgraph`. **No API key.**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Your setup, and a first graph

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print(f"Python {sys.version.split()[0]}")
for pkg in ("langgraph", "langchain-core", "langchain", "pydantic"):
    try:
        print(f"  {pkg:<15} {version(pkg)}")
    except PackageNotFoundError:
        print(f"  {pkg:<15} NOT INSTALLED - pip install {pkg}")

# The smallest possible LangGraph program: one node that changes the state.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):          # the typing module - a prerequisite
    text: str

def shout(state: State) -> dict:
    return {"text": state["text"].upper() + "!"}

graph = StateGraph(State)
graph.add_node("shout", shout)
graph.add_edge(START, "shout")
graph.add_edge("shout", END)
app = graph.compile()

print(app.invoke({"text": "langgraph is ready"}))
# {'text': 'LANGGRAPH IS READY!'}

In [ ]:
out = app.invoke({"text": "hello"})
assert out == {"text": "HELLO!"}
print("nodes:", list(app.get_graph().nodes))
assert list(app.get_graph().nodes) == ["__start__", "shout", "__end__"]

`START` and `END` are real nodes in the compiled graph. Every graph you build in
this track runs from one to the other.

## Part B — TypedDict

A `TypedDict` is a dictionary with named, typed keys. It documents the state.
It does **not** check anything at run time.

In [ ]:
from typing import TypedDict

class JobState(TypedDict):
    role: str
    applications: int

s: JobState = {"role": "backend engineer", "applications": "two"}   # wrong type
print(s)          # no error: TypedDict is a hint for you and your editor, not a check
assert s["applications"] == "two"

## Part C — Pydantic checks, TypedDict does not

In [ ]:
from pydantic import BaseModel, ValidationError

class JobModel(BaseModel):
    role: str
    applications: int

print(JobModel(role="backend engineer", applications="2"))   # "2" is converted to 2
try:
    JobModel(role="backend engineer", applications="two")
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])
    caught = True
assert caught

LangGraph accepts either as a state schema. Use a Pydantic model when you want
bad data stopped at the door.

## Part D — asyncio

LLM calls are slow and mostly spent waiting. `asyncio` lets several waits
overlap. The sleeps below stand in for two model calls.

In [ ]:
import asyncio, time

async def fake_llm_call(name, seconds=0.3):
    await asyncio.sleep(seconds)
    return name

async def one_after_another():
    return [await fake_llm_call("a"), await fake_llm_call("b")]

async def at_the_same_time():
    return await asyncio.gather(fake_llm_call("a"), fake_llm_call("b"))

for f in (one_after_another, at_the_same_time):
    t = time.perf_counter()
    await f()
    print(f"{f.__name__:<18} {time.perf_counter() - t:.2f} s")

The two waits overlap, so the second takes about half as long. Parallel
branches in LangGraph (lesson 007) rely on the same idea.

## What to take away

- LangGraph is installed and a graph runs: `START -> shout -> END`.
- `TypedDict` describes state; Pydantic also **checks** it.
- `asyncio` overlaps waiting — useful when every step calls a slow model.

## Exercises

1. Add a second node, `exclaim`, after `shout`, that adds two more `!`. What
   does `get_graph().nodes` list now?
2. Use `JobModel` as the graph's state instead of the `TypedDict`. What happens
   when you invoke it with `applications="two"`?
3. Time three fake calls with `gather`. Is it still about 0.3 seconds?